# 12 — CPython Internals
**References:** CPython source (github.com/python/cpython) · Louie (2022) *CPython Internals* · Beazley *Python Essential Reference* App. A

## Narrative thread
```
Source -> AST -> Bytecode -> Frames -> Eval loop -> C API -> Extending Python
```

## 1. Python compilation pipeline

When CPython executes `python script.py`, it goes through:

```
Source (.py)
  → Tokenizer (tokenize module)
  → Parser → AST (ast module)
  → Symbol table
  → Bytecode compiler → Code object (.pyc)
  → CPython eval loop (ceval.c)
```

All of this is accessible from Python itself.

In [ ]:
import ast
import dis
import sys
import tokenize
import io
import types
import inspect
import ctypes

# ── Step 1: Tokenization ──────────────────────────────────────────────────
source = 'result = x**2 + y**2'
print('=== Tokens ===')
tokens = tokenize.generate_tokens(io.StringIO(source).readline)
for tok in tokens:
    if tok.type not in (tokenize.ENCODING, tokenize.ENDMARKER, tokenize.NEWLINE):
        print(f'  {tokenize.tok_name[tok.type]:10} {tok.string!r}')

print()
# ── Step 2: AST ──────────────────────────────────────────────────────────
print('=== AST ===')
tree = ast.parse(source)
print(ast.dump(tree, indent=2)[:600], '...')

print()
# ── Step 3: Bytecode (dis module) ────────────────────────────────────────
print('=== Bytecode ===')

def example(x, y):
    result = x**2 + y**2
    return result

dis.dis(example)

In [ ]:
# ── Code object: what CPython stores for each function ───────────────────
print('=== Code object attributes ===')
code = example.__code__
attrs = [
    ('co_name',        code.co_name),
    ('co_filename',    code.co_filename),
    ('co_firstlineno', code.co_firstlineno),
    ('co_argcount',    code.co_argcount),
    ('co_varnames',    code.co_varnames),    # local variables
    ('co_consts',      code.co_consts),      # constants
    ('co_names',       code.co_names),       # global names
    ('co_stacksize',   code.co_stacksize),   # max stack depth
    ('co_flags',       hex(code.co_flags)),  # various flags
]
for attr, val in attrs:
    print(f'  {attr:20} {val!r}')

print()
# ── Bytecode instruction objects ─────────────────────────────────────────
print('=== Instruction objects ===')
for instr in dis.get_instructions(example):
    print(f'  offset={instr.offset:3} {instr.opname:20} {instr.argrepr}')

In [ ]:
# ── Frame objects and the call stack ─────────────────────────────────────
def get_current_frame():
    return sys._getframe()

def show_stack():
    frame = sys._getframe()
    print('=== Call stack (innermost first) ===')
    depth = 0
    while frame:
        print(f'  [{depth}] {frame.f_code.co_name}() in {frame.f_code.co_filename}:{frame.f_lineno}')
        print(f'       locals: {list(frame.f_locals.keys())}')
        frame = frame.f_back
        depth += 1
        if depth > 5: break

def inner():
    x = 42
    show_stack()

def outer():
    y = 'hello'
    inner()

outer()

print()
# ── CPython integer caching (-5 to 256) ──────────────────────────────────
print('=== CPython implementation details ===')

# Small ints are cached — same object
a, b = 256, 256
print(f'256 is 256: {a is b}  (cached)')

a, b = 257, 257
print(f'257 is 257: {a is b}  (not always cached — CPython-specific optimization)')

# String interning
s1 = 'hello_world'  # identifier-like — may be interned
s2 = 'hello_world'
s3 = 'hello' + '_world'  # explicit concatenation — may not be interned
print(f"'hello_world' is 'hello_world':   {s1 is s2}  (interned at compile time)")
print(f"intern(s3) is s1: {sys.intern(s3) is s1}  (force interning)")

In [ ]:
# ── AST manipulation: write a simple optimizer ────────────────────────────
class ConstantFolder(ast.NodeTransformer):
    """Fold constant arithmetic at the AST level."""

    def visit_BinOp(self, node):
        self.generic_visit(node)  # process children first
        if isinstance(node.left, ast.Constant) and isinstance(node.right, ast.Constant):
            try:
                ops = {
                    ast.Add:  lambda a,b: a+b,
                    ast.Sub:  lambda a,b: a-b,
                    ast.Mult: lambda a,b: a*b,
                    ast.Div:  lambda a,b: a/b,
                    ast.Pow:  lambda a,b: a**b,
                }
                op_fn = ops.get(type(node.op))
                if op_fn:
                    result = op_fn(node.left.value, node.right.value)
                    return ast.Constant(value=result)
            except Exception:
                pass
        return node

source_expr = '2 * (3 + 4) * 5'
tree   = ast.parse(source_expr, mode='eval')
folded = ConstantFolder().visit(tree)
ast.fix_missing_locations(folded)

print(f'Original: {source_expr}')
print(f'Folded:   {ast.dump(folded.body)}')
print(f'Result:   {eval(compile(folded, "<ast>", "eval"))}')

print()
# ── Reference count (CPython implementation detail) ───────────────────────
import ctypes

def ref_count(obj):
    """Get reference count of an object (CPython only)."""
    return sys.getrefcount(obj) - 1  # subtract 1 for the getrefcount call itself

x = 'a unique string that is not interned hopefully 12345'
print(f'ref count of x (1 ref):   {ref_count(x)}')
y = x
print(f'ref count of x (2 refs):  {ref_count(x)}')
lst = [x, x, x]
print(f'ref count of x (5 refs):  {ref_count(x)}')
del y, lst
print(f'ref count of x (1 ref):   {ref_count(x)}')

## 2. Extending Python with C (overview)

For performance-critical code, Python provides several mechanisms to call C:

| Mechanism | Best for |
|---|---|
| `ctypes` | Call existing shared libraries without compilation |
| `cffi` | More Pythonic, better for larger C APIs |
| `Cython` | Write Python-like code, compile to C — great for NumPy integration |
| CPython C API | Full control; write `.so` modules; highest performance |
| `pybind11` | C++11 bindings with automatic type conversion |

```python
# ctypes: call libc directly
import ctypes
libc = ctypes.CDLL(None)       # load libc
libc.printf(b'Hello from C!\n')
```

## 3. Key takeaways

| Concept | Statement |
|---|---|
| Code object | Immutable bytecode container; `__code__` attribute on functions |
| `dis.dis()` | Disassemble any function or code string to bytecode |
| Frame object | Runtime state of one function call; `sys._getframe()` |
| Int caching | CPython caches `-5` to `256`; `is` on ints outside range is undefined |
| `ast` module | Parse, inspect, and transform Python source at the AST level |
| `sys.getrefcount()` | See CPython's reference count; result is +1 (arg reference) |
| `ctypes` | Zero-compilation C interop for existing shared libraries |

> **End of course.** You now have deep knowledge of Python's object model, metaprogramming, concurrency, async programming, memory management, the type system, testing, packaging, and CPython internals.